# 9.4 线性探针与引导向量 / Linear probes and steering vectors

训练一个线性分类器去读取中间表示中的信息，用对照实验判断结果是否可信，再沿找到的方向改写表示，检验模型是否使用了它。

Training a linear classifier to read information from intermediate representations, judging by control experiments whether the result can be trusted, and rewriting the representation along the direction found to test whether the model uses it.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 训练探针并做对照实验 / Train a probe and run the controls

取出 160 个城市在 GPT-2 small 各层最后一个 token 上的状态，用分层 5 折交叉验证评估一个预测大洲的线性探针，并完成三个对照，即打乱标签、随机权重的模型和训练集准确率。测试先在合成数据上检查评估函数，再检查真实实验的几项结论。请阅读打印出的全部结果，包括 token 数这一混淆因素、对正则强度的敏感性以及模型自身的回答。需要 scikit-learn，运行时间约 50 秒。

Take the states of 160 cities at the last token in every layer of GPT-2 small, evaluate with stratified 5-fold cross-validation a linear probe that predicts the continent, and run three controls, namely shuffled labels, a model with random weights and accuracy on the training set. The tests check the evaluation function on synthetic data and then several conclusions of the real experiment. Read all printed results, including the confound of the token count, the sensitivity to the regularization strength and the answers of the model itself. scikit-learn is required, and the run takes about 50 seconds.

需要联网下载模型。/ Downloads a model.

In [ ]:
import warnings
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# 160 cities, 40 per label. The labels are the task of the probe.
CITIES = {
    "Europe": [
        "London", "Paris", "Berlin", "Madrid", "Rome", "Vienna", "Prague", "Warsaw",
        "Budapest", "Amsterdam", "Brussels", "Lisbon", "Dublin", "Athens", "Stockholm", "Oslo",
        "Copenhagen", "Helsinki", "Zurich", "Munich", "Hamburg", "Milan", "Naples", "Barcelona",
        "Lyon", "Marseille", "Manchester", "Edinburgh", "Glasgow", "Geneva", "Frankfurt", "Venice",
        "Florence", "Seville", "Porto", "Krakow", "Bucharest", "Sofia", "Belgrade", "Zagreb",
    ],
    "Asia": [
        "Tokyo", "Beijing", "Shanghai", "Seoul", "Bangkok", "Jakarta", "Manila", "Hanoi",
        "Singapore", "Mumbai", "Delhi", "Kolkata", "Chennai", "Bangalore", "Karachi", "Lahore",
        "Dhaka", "Kathmandu", "Colombo", "Tehran", "Baghdad", "Riyadh", "Dubai", "Doha",
        "Kabul", "Tashkent", "Osaka", "Kyoto", "Guangzhou", "Shenzhen", "Wuhan", "Chengdu",
        "Taipei", "Busan", "Yangon", "Islamabad", "Hyderabad", "Nanjing", "Tianjin", "Pyongyang",
    ],
    "Africa": [
        "Cairo", "Lagos", "Nairobi", "Johannesburg", "Accra", "Dakar", "Casablanca", "Tunis",
        "Algiers", "Khartoum", "Kampala", "Kigali", "Luanda", "Maputo", "Harare", "Lusaka",
        "Kinshasa", "Abuja", "Tripoli", "Mogadishu", "Durban", "Pretoria", "Alexandria", "Marrakech",
        "Bamako", "Freetown", "Monrovia", "Windhoek", "Gaborone", "Abidjan", "Kano", "Mombasa",
        "Rabat", "Niamey", "Lilongwe", "Antananarivo", "Ouagadougou", "Conakry", "Asmara", "Djibouti",
    ],
    "Americas": [
        "Chicago", "Boston", "Houston", "Dallas", "Seattle", "Denver", "Miami", "Atlanta",
        "Detroit", "Philadelphia", "Toronto", "Montreal", "Vancouver", "Ottawa", "Havana", "Bogota",
        "Lima", "Santiago", "Caracas", "Quito", "Montevideo", "Brasilia", "Guadalajara", "Monterrey",
        "Phoenix", "Portland", "Calgary", "Winnipeg", "Medellin", "Recife", "Salvador", "Curitiba",
        "Valparaiso", "Cusco", "Tijuana", "Pittsburgh", "Baltimore", "Minneapolis", "Nashville", "Honolulu",
    ],
}
CONTINENTS = list(CITIES)


def city_dataset():
    """Returns (names, y): the 160 city names and their labels 0..3 as a numpy array."""
    names = [c for k in CONTINENTS for c in CITIES[k]]
    y = np.array([i for i, k in enumerate(CONTINENTS) for _ in CITIES[k]])
    return names, y


def load_models():
    """GPT-2 small, its tokenizer, and a GPT-2 small with random weights as a control."""
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    torch.manual_seed(0)
    random_model = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained("gpt2")).eval()
    return model, tok, random_model


@torch.no_grad()
def last_token_states(model, tok, texts):
    """Hidden states at the last token of every text. Returns a numpy array (N, 13, 768).

    Index 0 is the embedding, index l the state after block l - 1. In the GPT-2 implementation of
    transformers the last index has the final LayerNorm applied, which does not matter for a probe.
    """
    # TODO 1: for every text run the model with output_hidden_states=True, take h[0, -1] from each
    #         of the 13 hidden states and stack. Stack over the texts and convert with .numpy().
    raise NotImplementedError


def make_probe(C):
    """Standardize the features, then multinomial logistic regression with L2 penalty 1 / C."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))


def probe_cv_accuracy(X, y, C=0.1, folds=5, seed=0):
    """Mean accuracy on held-out examples over stratified k-fold cross-validation. X (N, d), y (N,)."""
    # TODO 2: StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y) yields index pairs.
    #         Fit make_probe(C) on the training part of each fold and score it on the held-out part.
    #         The scaler is inside the pipeline, so it is fitted on the training part only.
    raise NotImplementedError


def probe_train_accuracy(X, y, C=0.1):
    """Accuracy on the same examples the probe was trained on."""
    # TODO 3: fit on (X, y) and score on (X, y)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import numpy as np
import torch

names, y = city_dataset()
assert len(names) == len(set(names)) == 160 and np.bincount(y).tolist() == [40, 40, 40, 40]

# 1. Synthetic data. A class signal spread over the dimensions is found, pure noise is not.
rng = np.random.default_rng(0)
ys = np.repeat(np.arange(4), 40)
noise = rng.normal(size=(160, 768)).astype(np.float32)
centres = rng.normal(size=(4, 768)).astype(np.float32)
signal = noise + 0.5 * centres[ys]                  # every class is shifted a little in all dimensions
assert probe_cv_accuracy(signal, ys) > 0.9
assert probe_cv_accuracy(noise, ys) < 0.45
# With more dimensions than examples a linear probe fits any labels on its training set.
assert probe_train_accuracy(noise, ys) == 1.0, "768 dimensions, 160 points: every labelling is separable"
assert probe_cv_accuracy(noise, ys, seed=0) == probe_cv_accuracy(noise, ys, seed=0)

# 2. Features from GPT-2.
model, tok, random_model = load_models()
texts = [f"The city of {c}" for c in names]
F = last_token_states(model, tok, texts)
assert F.shape == (160, 13, 768) and F.dtype == np.float32
with torch.no_grad():
    ids = tok(texts[0], return_tensors="pt").input_ids
    ref = model(ids, output_hidden_states=True).hidden_states
assert np.allclose(F[0, 5], ref[5][0, -1].numpy(), atol=1e-4)

# 3. The probe and three controls.
real = [probe_cv_accuracy(F[:, l], y) for l in range(13)]
shuffled_y = np.random.default_rng(0).permutation(y)
shuf = [probe_cv_accuracy(F[:, l], shuffled_y) for l in range(13)]
FR = last_token_states(random_model, tok, texts)
rand = [probe_cv_accuracy(FR[:, l], y) for l in range(13)]
assert max(real) > 0.8, "the continent can be read from the last token of the city name"
assert max(shuf) < 0.4, "shuffled labels cannot be predicted for held-out cities"
assert real[0] > 0.6, "much of it is in the embedding already"
assert max(rand) < min(real), "a model with random weights does worse at every layer"
assert probe_train_accuracy(F[:, 6], shuffled_y) == 1.0

print("held-out accuracy of a linear probe for the continent, chance = 0.25")
print("layer            " + " ".join(f"{l:>5}" for l in range(13)))
print("GPT-2            " + " ".join(f"{a:>5.2f}" for a in real))
print("shuffled labels  " + " ".join(f"{a:>5.2f}" for a in shuf))
print("random weights   " + " ".join(f"{a:>5.2f}" for a in rand))
print(f"training accuracy on shuffled labels at layer 6: {probe_train_accuracy(F[:, 6], shuffled_y):.2f}")

# Why the random model is above chance: the number of tokens of a name differs between the labels.
n_tok = np.array([len(tok(" " + c).input_ids) for c in names])
print("mean number of tokens per city name: " + ", ".join(f"{k} {n_tok[y == i].mean():.2f}" for i, k in enumerate(CONTINENTS)))
print(f"probe on the token count alone: {probe_cv_accuracy(n_tok[:, None].astype(np.float32), y):.2f}")

# Sensitivity to the regularization strength and to the fold assignment.
print("layer 9, C = 0.01, 0.1, 1, 10: " + ", ".join(f"{probe_cv_accuracy(F[:, 9], y, C=c):.2f}" for c in (0.01, 0.1, 1, 10)))
seeds = [probe_cv_accuracy(F[:, 9], y, seed=s) for s in range(5)]
print(f"layer 9, five different fold assignments: {min(seeds):.2f} to {max(seeds):.2f}")

# What the model itself says.
cand = [tok(" " + w).input_ids[0] for w in ("Europe", "Asia", "Africa", "America")]
with torch.no_grad():
    pred = np.array([model(tok(f"The city of {c} is in", return_tensors="pt").input_ids).logits[0, -1][cand].argmax().item() for c in names])
print(f"accuracy of the model's own answer after 'The city of X is in', restricted to four continent tokens: {(pred == y).mean():.2f}"
      + " (" + ", ".join(f"{k} {(pred[y == i] == i).mean():.2f}" for i, k in enumerate(CONTINENTS)) + ")")

<details><summary><b>提示 1 / Hint 1</b></summary>

`for train, test in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y)` 给出每一折的下标。

`for train, test in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y)` yields the indices of every fold.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

标准化已经放在 `make_probe` 的流水线里，对每一折分别 `fit`，测试部分就不会参与均值和方差的估计。

Standardization is already part of the pipeline of `make_probe`. Fitting it separately for every fold keeps the test part out of the estimate of mean and variance.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import warnings
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

# 160 cities, 40 per label. The labels are the task of the probe.
CITIES = {
    "Europe": [
        "London", "Paris", "Berlin", "Madrid", "Rome", "Vienna", "Prague", "Warsaw",
        "Budapest", "Amsterdam", "Brussels", "Lisbon", "Dublin", "Athens", "Stockholm", "Oslo",
        "Copenhagen", "Helsinki", "Zurich", "Munich", "Hamburg", "Milan", "Naples", "Barcelona",
        "Lyon", "Marseille", "Manchester", "Edinburgh", "Glasgow", "Geneva", "Frankfurt", "Venice",
        "Florence", "Seville", "Porto", "Krakow", "Bucharest", "Sofia", "Belgrade", "Zagreb",
    ],
    "Asia": [
        "Tokyo", "Beijing", "Shanghai", "Seoul", "Bangkok", "Jakarta", "Manila", "Hanoi",
        "Singapore", "Mumbai", "Delhi", "Kolkata", "Chennai", "Bangalore", "Karachi", "Lahore",
        "Dhaka", "Kathmandu", "Colombo", "Tehran", "Baghdad", "Riyadh", "Dubai", "Doha",
        "Kabul", "Tashkent", "Osaka", "Kyoto", "Guangzhou", "Shenzhen", "Wuhan", "Chengdu",
        "Taipei", "Busan", "Yangon", "Islamabad", "Hyderabad", "Nanjing", "Tianjin", "Pyongyang",
    ],
    "Africa": [
        "Cairo", "Lagos", "Nairobi", "Johannesburg", "Accra", "Dakar", "Casablanca", "Tunis",
        "Algiers", "Khartoum", "Kampala", "Kigali", "Luanda", "Maputo", "Harare", "Lusaka",
        "Kinshasa", "Abuja", "Tripoli", "Mogadishu", "Durban", "Pretoria", "Alexandria", "Marrakech",
        "Bamako", "Freetown", "Monrovia", "Windhoek", "Gaborone", "Abidjan", "Kano", "Mombasa",
        "Rabat", "Niamey", "Lilongwe", "Antananarivo", "Ouagadougou", "Conakry", "Asmara", "Djibouti",
    ],
    "Americas": [
        "Chicago", "Boston", "Houston", "Dallas", "Seattle", "Denver", "Miami", "Atlanta",
        "Detroit", "Philadelphia", "Toronto", "Montreal", "Vancouver", "Ottawa", "Havana", "Bogota",
        "Lima", "Santiago", "Caracas", "Quito", "Montevideo", "Brasilia", "Guadalajara", "Monterrey",
        "Phoenix", "Portland", "Calgary", "Winnipeg", "Medellin", "Recife", "Salvador", "Curitiba",
        "Valparaiso", "Cusco", "Tijuana", "Pittsburgh", "Baltimore", "Minneapolis", "Nashville", "Honolulu",
    ],
}
CONTINENTS = list(CITIES)


def city_dataset():
    """Returns (names, y): the 160 city names and their labels 0..3 as a numpy array."""
    names = [c for k in CONTINENTS for c in CITIES[k]]
    y = np.array([i for i, k in enumerate(CONTINENTS) for _ in CITIES[k]])
    return names, y


def load_models():
    """GPT-2 small, its tokenizer, and a GPT-2 small with random weights as a control."""
    tok = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    torch.manual_seed(0)
    random_model = AutoModelForCausalLM.from_config(AutoConfig.from_pretrained("gpt2")).eval()
    return model, tok, random_model


@torch.no_grad()
def last_token_states(model, tok, texts):
    """Hidden states at the last token of every text. Returns a numpy array (N, 13, 768).

    Index 0 is the embedding, index l the state after block l - 1. In the GPT-2 implementation of
    transformers the last index has the final LayerNorm applied, which does not matter for a probe.
    """
    rows = []
    for t in texts:
        hs = model(tok(t, return_tensors="pt").input_ids, output_hidden_states=True).hidden_states
        rows.append(torch.stack([h[0, -1] for h in hs]))
    return torch.stack(rows).numpy()


def make_probe(C):
    """Standardize the features, then multinomial logistic regression with L2 penalty 1 / C."""
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))


def probe_cv_accuracy(X, y, C=0.1, folds=5, seed=0):
    """Mean accuracy on held-out examples over stratified k-fold cross-validation. X (N, d), y (N,)."""
    accs = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for train, test in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y):
            probe = make_probe(C).fit(X[train], y[train])
            accs.append(probe.score(X[test], y[test]))
    return float(np.mean(accs))


def probe_train_accuracy(X, y, C=0.1):
    """Accuracy on the same examples the probe was trained on."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(make_probe(C).fit(X, y).score(X, y))
```

</details>

## 2. 用均值差构造引导向量 / Build a steering vector from a mean difference

计算亚洲城市与欧洲城市在某一层的平均状态之差，并实现在城市名最后一个 token 的位置上加入该向量后测量 Asia 与 Europe 的 logit 差。方向用一半城市计算，在另一半城市上检验。测试检查加入零向量不改变输出、第 6 层加入 2 倍向量使多数留出城市发生转变，而同样长度的随机方向没有这种效果。请查看打印出的各层、各强度的结果，以及与逻辑回归探针方向的比较。运行时间约 100 秒。

Compute the difference between the mean states of Asian and of European cities at some layer, and implement measuring the logit difference between Asia and Europe with the vector added at the position of the last token of the city name. The direction is computed from one half of the cities and tested on the other half. The tests check that adding a zero vector leaves the output unchanged, that adding twice the vector at layer 6 turns most held-out cities, and that a random direction of the same length has no such effect. Look at the printed results for the layers and strengths and at the comparison with the direction of a logistic regression probe. The run takes about 100 seconds.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch

TEMPLATE = "The city of {} is in"


def prompt(tok, city):
    """Returns (ids (1, T), position of the last token of the city name)."""
    ids = tok(TEMPLATE.format(city), return_tensors="pt").input_ids
    pos = len(tok("The city of " + city).input_ids) - 1
    return ids, pos


@torch.no_grad()
def city_state(model, tok, city, layer):
    """Output of block `layer` at the last token of the city name. Returns (768,)."""
    # TODO 1: hidden_states[layer + 1] is the output of block `layer`
    raise NotImplementedError


def mean_difference(model, tok, cities_to, cities_from, layer):
    """Mean state of cities_to minus mean state of cities_from."""
    # TODO 2
    raise NotImplementedError


@torch.no_grad()
def steered_logit_diff(model, tok, city, layer, vec, token_to, token_from):
    """logit(token_to) - logit(token_from) for the next token, with vec added to the output of block
    `layer` at the last token of the city name. vec None means no intervention."""
    ids, pos = prompt(tok, city)
    # TODO 3: if vec is given, a forward hook on model.transformer.h[layer] that adds vec at [0, pos]
    #         (clone first, and return a tuple if the block output is a tuple)
    # TODO 4: run, remove the hook, return the logit difference at the last position as a float
    raise NotImplementedError

In [ ]:
# 检查 / checks
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression

model, tok, _ = load_models()
EU, AS = tok(" Europe").input_ids[0], tok(" Asia").input_ids[0]
eu, asia = CITIES["Europe"], CITIES["Asia"]
train_eu, test_eu, train_as, test_as = eu[:20], eu[20:], asia[:20], asia[20:]     # the direction comes from one half, the test uses the other

# 1. Positions and states.
ids, pos = prompt(tok, "Paris")
assert tok.decode(ids[0, pos]) == " Paris"
ids, pos = prompt(tok, "Antananarivo")
assert tok.decode(ids[0, :pos + 1]).endswith("Antananarivo") and not tok.decode(ids[0, :pos]).endswith("Antananarivo")
s = city_state(model, tok, "Paris", 6)
with torch.no_grad():
    ref = model(prompt(tok, "Paris")[0], output_hidden_states=True).hidden_states[7][0, 3]
assert s.shape == (768,) and torch.allclose(s, ref, atol=1e-4)

# 2. The direction.
v = mean_difference(model, tok, train_as, train_eu, 6)
assert v.shape == (768,)
assert torch.allclose(v, -mean_difference(model, tok, train_eu, train_as, 6), atol=1e-4)

# 3. No vector and a zero vector change nothing, and the hook is removed.
n_hooks = sum(len(b._forward_hooks) for b in model.transformer.h)
base = steered_logit_diff(model, tok, "Berlin", 6, None, AS, EU)
assert abs(steered_logit_diff(model, tok, "Berlin", 6, torch.zeros(768), AS, EU) - base) < 1e-4
assert sum(len(b._forward_hooks) for b in model.transformer.h) == n_hooks
assert base < 0, "without intervention the model prefers Europe for Berlin"

# 4. Steering held-out European cities toward Asia.
def sweep(layer, vec, alpha):
    d = [steered_logit_diff(model, tok, c, layer, alpha * vec, AS, EU) for c in test_eu]
    return float(np.mean(d)), float(np.mean([x > 0 for x in d]))
base_eu = float(np.mean([steered_logit_diff(model, tok, c, 6, None, AS, EU) for c in test_eu]))
base_as = float(np.mean([steered_logit_diff(model, tok, c, 6, None, AS, EU) for c in test_as]))
g = torch.Generator().manual_seed(0)
rand = torch.randn(768, generator=g); rand = rand / rand.norm() * v.norm()
m2, f2 = sweep(6, v, 2.0)
mr, fr = sweep(6, rand, 2.0)
assert f2 >= 0.8, "twice the mean difference flips most held-out cities"
assert fr <= 0.1 and mr < m2 - 2, "a random direction of the same length does not"

print(f"logit(' Asia') - logit(' Europe') without intervention: held-out European cities {base_eu:+.2f}, held-out Asian cities {base_as:+.2f}")
print("layer  alpha   mean difference   share flipped to Asia   |   random direction of equal norm")
results = {}
for layer in (0, 2, 4, 6, 8, 10):
    vl = mean_difference(model, tok, train_as, train_eu, layer)
    rl = torch.randn(768, generator=g); rl = rl / rl.norm() * vl.norm()
    for alpha in (1.0, 2.0, 4.0):
        m, f = sweep(layer, vl, alpha); mr_, fr_ = sweep(layer, rl, alpha)
        results[(layer, alpha)] = f
        print(f"{layer:>5}  {alpha:>5.0f}   {m:>+15.2f}   {f:>21.2f}   |   {mr_:>+6.2f}  {fr_:.2f}")
assert results[(10, 2.0)] < 0.5, "at layer 10 the city position no longer feeds the answer"

# 5. The weight vector of a logistic-regression probe as steering direction.
X = torch.stack([city_state(model, tok, c, 6) for c in train_eu + train_as]).numpy()
lab = np.array([0] * 20 + [1] * 20)
mu, sd = X.mean(0), X.std(0) + 1e-6
w = torch.tensor(LogisticRegression(C=0.1, max_iter=5000).fit((X - mu) / sd, lab).coef_[0] / sd, dtype=torch.float32)
print(f"layer 6: cosine between the probe weight vector and the mean difference {torch.cosine_similarity(w, v, 0).item():.2f}")
wn = w / w.norm() * v.norm()
for alpha in (1.0, 2.0, 4.0):
    m, f = sweep(6, wn, alpha)
    print(f"  probe direction scaled to the same norm, alpha {alpha:.0f}: mean {m:+.2f}, share flipped {f:.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`hidden_states[layer + 1]` 是第 `layer` 个块的输出，下标 0 是嵌入。

`hidden_states[layer + 1]` is the output of block `layer`, index 0 being the embedding.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

钩子里写 `x[0, pos] += vec`，只改动城市名最后一个 token 的位置。块的输出是元组时返回 `(x,) + tuple(out[1:])`。

Inside the hook write `x[0, pos] += vec`, changing the position of the last token of the city name alone. Return `(x,) + tuple(out[1:])` when the output of the block is a tuple.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch

TEMPLATE = "The city of {} is in"


def prompt(tok, city):
    """Returns (ids (1, T), position of the last token of the city name)."""
    ids = tok(TEMPLATE.format(city), return_tensors="pt").input_ids
    pos = len(tok("The city of " + city).input_ids) - 1
    return ids, pos


@torch.no_grad()
def city_state(model, tok, city, layer):
    """Output of block `layer` at the last token of the city name. Returns (768,)."""
    ids, pos = prompt(tok, city)
    return model(ids, output_hidden_states=True).hidden_states[layer + 1][0, pos]


def mean_difference(model, tok, cities_to, cities_from, layer):
    """Mean state of cities_to minus mean state of cities_from."""
    a = torch.stack([city_state(model, tok, c, layer) for c in cities_to]).mean(0)
    b = torch.stack([city_state(model, tok, c, layer) for c in cities_from]).mean(0)
    return a - b


@torch.no_grad()
def steered_logit_diff(model, tok, city, layer, vec, token_to, token_from):
    """logit(token_to) - logit(token_from) for the next token, with vec added to the output of block
    `layer` at the last token of the city name. vec None means no intervention."""
    ids, pos = prompt(tok, city)
    handle = None
    if vec is not None:
        def hook(module, args, out):
            x = (out[0] if isinstance(out, tuple) else out).clone()
            x[0, pos] += vec
            return (x,) + tuple(out[1:]) if isinstance(out, tuple) else x
        handle = model.transformer.h[layer].register_forward_hook(hook)
    try:
        logits = model(ids).logits[0, -1]
    finally:
        if handle is not None:
            handle.remove()
    return (logits[token_to] - logits[token_from]).item()
```

</details>

## 3. 探针的统计与几何 / Statistics and geometry of probes

在纸上完成。
(a) 准确率 $\hat{p}$ 由 $n$ 个独立的测试样本估计，它的标准误为 $\sqrt{\hat{p}(1 - \hat{p})/n}$。计算 $n = 160$、$\hat{p} = 0.85$ 时的标准误。练习 1 中第 9 层的准确率是 0.83，第 10 层是 0.86，能否认为第 10 层的表示更好？
(b) 取 $d = 2$，三个点 $x_1 = (1, 0)$，$x_2 = (0, 1)$，$x_3 = (1, 1)$。说明存在一种标签分配，使得没有任何过原点的直线能把它们正确分开。这与推导 2 是否矛盾？
(c) 两类样本在某一层的均值为 $\mu_A$ 和 $\mu_B$，引导向量为 $v = \mu_B - \mu_A$。一个 A 类样本的状态为 $h$。证明加上 $\alpha v$ 之后，它在 $v$ 方向上的投影增加了 $\alpha \lVert v \rVert$，并说明为什么 $\alpha = 1$ 时只有一部分样本越过了两类的中点。
(d) 有人训练了一个探针，从模型的中间表示预测“当前上下文中是否包含工具返回的结果”，留出准确率为 0.99，并据此认为模型内部有一个表示工具结果的特征。指出这一结论可能存在的问题，并设计两个对照。

Work on paper.
(a) An accuracy $\hat{p}$ estimated from $n$ independent test samples has the standard error $\sqrt{\hat{p}(1 - \hat{p})/n}$. Compute the standard error for $n = 160$ and $\hat{p} = 0.85$. In exercise 1 the accuracy is 0.83 at layer 9 and 0.86 at layer 10. Can the representation of layer 10 be considered better?
(b) Take $d = 2$ and the three points $x_1 = (1, 0)$, $x_2 = (0, 1)$, $x_3 = (1, 1)$. Show that there is an assignment of labels for which no line through the origin separates them correctly. Does this contradict derivation 2?
(c) Two classes have the means $\mu_A$ and $\mu_B$ at some layer, and the steering vector is $v = \mu_B - \mu_A$. A sample of class A has the state $h$. Prove that adding $\alpha v$ increases its projection onto the direction of $v$ by $\alpha \lVert v \rVert$, and explain why with $\alpha = 1$ only a part of the samples crosses the midpoint between the two classes.
(d) Someone trains a probe that predicts from the intermediate representations of a model whether the current context contains a result returned by a tool, obtains a held-out accuracy of 0.99 and concludes that the model has an internal feature representing tool results. Point out possible problems with this conclusion and design two controls.

<details><summary><b>提示 1 / Hint 1</b></summary>

(b) $x_3 = x_1 + x_2$，考虑 $w^\top x_3$ 的符号。

(b) $x_3 = x_1 + x_2$, so consider the sign of $w^\top x_3$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 投影是 $h^\top v / \lVert v \rVert$。

(c) The projection is $h^\top v / \lVert v \rVert$.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

(d) 想一想包含工具结果的上下文在表面上与不包含的有什么不同。

(d) Consider how a context that contains a tool result differs on the surface from one that does not.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $\sqrt{0.85 \times 0.15 / 160} = \sqrt{0.000797} \approx 0.028$。两层相差 0.03，约为一个标准误，在通常的标准下不能认为有差别。两层使用的是同一批城市，严格的比较应当用配对检验，但在这个样本量下结论不会改变。要比较相邻的层，需要更多的样本。

(b) 取标签 $t = (+1, +1, -1)$。需要 $w^\top x_1 > 0$ 且 $w^\top x_2 > 0$，于是 $w^\top x_3 = w^\top x_1 + w^\top x_2 > 0$，与 $t_3 = -1$ 矛盾。这与推导 2 不矛盾，推导 2 要求 $n \le d$ 且各点线性无关，这里 $n = 3 > d = 2$。样本数超过维数之后，随机标签不再保证可以分开，探针的训练准确率才开始带有信息。

(c) 加入之前的投影是 $h^\top v / \lVert v \rVert$，加入之后是 $(h + \alpha v)^\top v / \lVert v \rVert = h^\top v / \lVert v \rVert + \alpha \lVert v \rVert$，增加了 $\alpha \lVert v \rVert$。两类均值的投影正好相差 $\lVert v \rVert$，中点在距 A 类均值 $\lVert v \rVert / 2$ 处。$\alpha = 1$ 时每个样本的投影移动 $\lVert v \rVert$。A 类样本本身分布在 $\mu_A$ 的周围，移动之后分布在 $\mu_B$ 的周围，其中投影原本低于 $\mu_A$ 超过 $\lVert v \rVert / 2$ 的样本仍未越过中点。此外模型的输出并不只取决于这一个方向上的投影。练习 2 中 $\alpha = 1$ 时有 50% 到 70% 的城市转变，$\alpha = 2$ 时为 90% 到 100%，与此相符。

(d) 可能的问题有三个。包含工具结果的上下文在表面上与不包含的差别很大，更长，并且含有特定的格式标记，探针可能只是读出了序列长度或某个标记 token 是否出现过。0.99 没有与任何基线比较。即使信息确实存在，也没有证据表明模型使用了它。

对照一，用表面特征训练探针，例如只用序列长度，或者用随机初始化的同结构模型的表示。如果这些基线也接近 0.99，探针的结果就没有说明什么。更好的办法是构造长度和格式都匹配的两组输入，例如一组包含真实的工具结果，另一组在同样的位置放一段格式相同但由用户写入的文字。

对照二，做干预。沿探针方向或两组的均值差方向改写表示，观察模型的行为是否改变，例如它是否更倾向于采信上下文中的内容。只有输出随之改变，才能说模型使用了这个方向。还可以检查探针在另一种工具格式上的迁移，如果换一种格式准确率就下降到基线，探针学到的更可能是格式。

---

(a) $\sqrt{0.85 \times 0.15 / 160} = \sqrt{0.000797} \approx 0.028$. The two layers differ by 0.03, about one standard error, which by usual standards does not count as a difference. Both layers use the same cities, so a strict comparison would use a paired test, though the conclusion does not change at this sample size. Comparing adjacent layers takes more samples.

(b) Take the labels $t = (+1, +1, -1)$. Required are $w^\top x_1 > 0$ and $w^\top x_2 > 0$, hence $w^\top x_3 = w^\top x_1 + w^\top x_2 > 0$, contradicting $t_3 = -1$. This does not contradict derivation 2, which requires $n \le d$ and linearly independent points, whereas here $n = 3 > d = 2$. Once the number of samples exceeds the dimension, random labels are no longer guaranteed to be separable, and the training accuracy of a probe begins to carry information.

(c) The projection before is $h^\top v / \lVert v \rVert$ and after it is $(h + \alpha v)^\top v / \lVert v \rVert = h^\top v / \lVert v \rVert + \alpha \lVert v \rVert$, an increase of $\alpha \lVert v \rVert$. The projections of the two class means differ by exactly $\lVert v \rVert$, and the midpoint lies $\lVert v \rVert / 2$ from the mean of class A. With $\alpha = 1$ the projection of every sample moves by $\lVert v \rVert$. The samples of class A are spread around $\mu_A$ and after the move around $\mu_B$, and those whose projection was more than $\lVert v \rVert / 2$ below $\mu_A$ have still not crossed the midpoint. The output of the model moreover does not depend on the projection onto this one direction alone. In exercise 2, 50% to 70% of the cities turn at $\alpha = 1$ and 90% to 100% at $\alpha = 2$, in line with this.

(d) There are three possible problems. Contexts that contain a tool result differ greatly on the surface from those that do not, being longer and containing specific formatting markers, so the probe may simply read the sequence length or whether some marker token occurred. The 0.99 is not compared with any baseline. And even if the information is present, nothing shows that the model uses it.

Control one. Train a probe on surface features, for example on the sequence length alone, or on the representations of a randomly initialized model of the same architecture. If these baselines also come close to 0.99, the probe result shows nothing. Better still is to construct two sets of inputs matched in length and format, for example one containing a real tool result and the other containing text of the same format at the same place that was written by the user.

Control two. Intervene. Rewrite the representation along the probe direction or along the mean difference of the two sets and observe whether the behaviour of the model changes, for example whether it becomes more inclined to trust the content of the context. Only if the output changes with it can the model be said to use this direction. Transfer of the probe to another tool format can be checked as well. If the accuracy falls to the baseline with another format, what the probe learned is more likely the format.

</details>